<a href="https://colab.research.google.com/github/adarshupadhyay06/gen_ai/blob/master/text_summeriser_using_lm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers datasets evaluate rouge_score nltk

In [4]:
import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [6]:
dataset = load_dataset("knkarthick/samsum")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [7]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def preprocess_function(examples):

    inputs = tokenizer(
        examples["dialogue"],
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=64,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [9]:
dataset_tokenized = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [11]:
rouge = evaluate.load("rouge")

In [12]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result

In [13]:
training_args = TrainingArguments(
    output_dir="samsum-model",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    eval_accumulation_steps=8
)

In [14]:
small_train = dataset_tokenized["train"].select(range(300))
small_eval = dataset_tokenized["validation"].select(range(100))

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=75, training_loss=42.406516520182294, metrics={'train_runtime': 429.1138, 'train_samples_per_second': 0.699, 'train_steps_per_second': 0.175, 'total_flos': 80303972425728.0, 'train_loss': 42.406516520182294, 'epoch': 1.0})

In [17]:
import evaluate
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(50):   # evaluate only 50 samples

    dialogue = dataset["validation"][i]["dialogue"]
    reference = dataset["validation"][i]["summary"]

    inputs = tokenizer(dialogue, return_tensors="pt", truncation=True).to(device)

    output = model.generate(inputs["input_ids"], max_length=60)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)

    predictions.append(pred)
    references.append(reference)

result = rouge.compute(predictions=predictions, references=references)

print(result)

{'rouge1': np.float64(0.0007547169811320755), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0007547169811320755), 'rougeLsum': np.float64(0.0007547169811320755)}


In [19]:
text = """
Amanda: Are we meeting today?
Jerry: Yes at 6 pm.
Amanda: Perfect see you there.
"""

iinputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

life me well only office me well only office me well only office me well only person me well only office me well only because me well only office me well only only


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
text = dataset["validation"][0]["dialogue"]

inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=5,
    length_penalty=2.0,
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

life good these while need show before power take good something off need May master first find many air good back huge take If good back see need May master me well only person good these while need show before power take good something off need May master me well
